# 04 — Validación end-to-end | Olist Lakehouse v2 · AUTO

Mantiene **todos los controles existentes** y agrega selección automática de la última validación aprobada como referencia. No hace falta copiar RUN_ID ni modificar widgets en cada corrida.

- `auto` **(predeterminado)**: compara con la última corrida aprobada con 25 snapshots. Si Bronze recibe datos duplicados y Silver/Gold se mantienen iguales, informa **IDEMPOTENCIA PASS**; si aparecen datos de negocio nuevos, registra **CAMBIO INCREMENTAL** y aplica las reconciliaciones habituales; si Bronze no cambia pero Silver/Gold sí, marca **FAIL**.
- `baseline`: registra snapshots sin comparar (modo manual conservado).
- `verify`: verificación manual contra un RUN_ID explícito, con alcance `silver_gold` o `all_25`.
- `incremental_observe`: prueba incremental manual, con expectativas configurables.

Los duplicados detectados se registran como WARNING: **Silver debe cortarlos y Gold no debe inflarse**.


1 — Configuración y parámetros
Mantener el mismo catálogo, esquema y zona horaria que los notebooks Silver/Gold. Se genera un identificador único para cada ejecución de esta validación.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    BooleanType, TimestampType, DecimalType
)
from datetime import datetime, timezone
from decimal import Decimal
import hashlib
import uuid

CATALOG = "workspace"
SCHEMA = "olist_lakehouse"
PREFIX = f"{CATALOG}.{SCHEMA}"
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Reconstruimos el selector MODE porque el archivo exportado conservaba
# metadatos de un widget antiguo que NO incluía "auto".
# Este validador se usa de forma automática por defecto.
try:
    dbutils.widgets.remove("mode")
except Exception:
    pass
dbutils.widgets.dropdown("mode", "auto", ["auto", "baseline", "verify", "incremental_observe"])
dbutils.widgets.text("baseline_run_id", "")
dbutils.widgets.text("job_run_id", "interactive")
dbutils.widgets.dropdown("expect_gold_change", "true", ["true", "false"])
dbutils.widgets.dropdown("idempotence_scope", "silver_gold", ["silver_gold", "all_25"])
dbutils.widgets.dropdown("expect_new_business_keys", "false", ["false", "true"])

MODE = dbutils.widgets.get("mode").strip()
BASELINE_ID = dbutils.widgets.get("baseline_run_id").strip()
JOB_RUN_ID = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
EXPECT_GOLD_CHANGE = dbutils.widgets.get("expect_gold_change") == "true"
IDEMPOTENCE_SCOPE = dbutils.widgets.get("idempotence_scope")
EXPECT_NEW_BUSINESS_KEYS = dbutils.widgets.get("expect_new_business_keys") == "true"
RUN_ID = str(uuid.uuid4())

BRONZE_SILVER = [
    "customers", "sellers", "category_translation", "orders",
    "order_items", "order_payments", "order_reviews", "products", "geolocation"
]
GOLD = [
    "gold_monthly_sales", "gold_monthly_category_share",
    "gold_monthly_top_products", "gold_monthly_category_state",
    "gold_delivery_performance", "gold_monthly_payment_methods",
    "gold_monthly_portfolio_evolution"
]

KEYS = {
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "geolocation": ["geolocation_zip_code_prefix"]
}
GOLD_KEYS = {
    "gold_monthly_sales": ["mes"],
    "gold_monthly_category_share": ["mes", "categoria_analitica"],
    "gold_monthly_top_products": ["mes", "ranking_mes"],
    "gold_monthly_category_state": ["mes", "estado", "categoria_pt"],
    "gold_delivery_performance": ["mes", "nivel_geografico", "estado"],
    "gold_monthly_payment_methods": ["mes", "metodo_pago"],
    "gold_monthly_portfolio_evolution": ["mes"]
}

TABLE_MAP = (
    [("BRONZE", f"{PREFIX}.bronze_{x}") for x in BRONZE_SILVER]
    + [("SILVER", f"{PREFIX}.silver_{x}") for x in BRONZE_SILVER]
    + [("GOLD", f"{PREFIX}.{x}") for x in GOLD]
)
AUDIT = f"{PREFIX}.pipeline_run_audit"
CHECK_DETAIL = f"{PREFIX}.pipeline_validation_checks"
SNAPSHOTS = f"{PREFIX}.pipeline_validation_snapshots"
REJECTS = f"{PREFIX}.silver_quality_rejects"
DUPLICATE_ALERTS = f"{PREFIX}.silver_duplicate_alerts"
BRONZE_AUDIT = f"{PREFIX}.bronze_ingestion_audit"

print(f"MODE={MODE} | RUN_ID={RUN_ID} | BASELINE_ID={BASELINE_ID or '(vacío)'}")
print(f"Se esperan {len(TABLE_MAP)} tablas de negocio (9 Bronze, 9 Silver, 7 Gold).")

MODE=auto | RUN_ID=0e605f32-3f9c-496e-9b4c-aaaa89842467 | BASELINE_ID=(vacío)
Se esperan 25 tablas de negocio (9 Bronze, 9 Silver, 7 Gold).


2 — Framework de pruebas
Se registran las verificaciones individualmente, con PASS/FAIL/ERROR e información de diagnóstico. Los fallos se consolidan al final, para que la auditoría quede guardada incluso cuando fallen controles.

In [0]:
checks = []
observations = {}


def check(name, passed, detail="", severity="CRITICAL"):
    checks.append((name, bool(passed), severity, str(detail)[:2000]))
    print(f"{'PASS' if passed else 'FAIL'} | {severity:<8} | {name} | {str(detail)[:160]}")
    return bool(passed)


def run_check(name, fn, severity="CRITICAL"):
    """fn debe devolver bool o la tupla (bool, detalle)."""
    try:
        result = fn()
        if isinstance(result, tuple):
            passed, detail = result
        else:
            passed, detail = result, ""
        return check(name, passed, detail, severity)
    except Exception as exc:
        check(name, False, f"{type(exc).__name__}: {exc}", severity)
        return False


def bad_rows(df, condition):
    return df.filter(condition).count()


def key_issues(df, keys):
    invalid = None
    for k in keys:
        part = F.col(k).isNull()
        if dict(df.dtypes)[k] == "string":
            part = part | (F.trim(F.col(k)) == "")
        if k in ("order_item_id", "payment_sequential"):
            part = part | (F.col(k) <= 0)
        invalid = part if invalid is None else (invalid | part)
    return bad_rows(df, invalid)


def duplicate_keys(df, keys):
    return df.groupBy(*keys).count().filter(F.col("count") > 1).count()


def compare_months(left, right, left_value, right_value, tolerance=Decimal("0.01")):
    """Full outer con presencia explícita: diferencia entre mes ausente y monto NULL."""
    a = left.select(
        "mes", F.col(left_value).cast("decimal(28,2)").alias("left_value"),
        F.lit(1).alias("left_present")
    )
    b = right.select(
        "mes", F.col(right_value).cast("decimal(28,2)").alias("right_value"),
        F.lit(1).alias("right_present")
    )
    joined = a.join(b, "mes", "full_outer")
    same = (
        F.col("left_present").isNotNull()
        & F.col("right_present").isNotNull()
        & (
            F.col("left_value").eqNullSafe(F.col("right_value"))
            | (
                F.col("left_value").isNotNull()
                & F.col("right_value").isNotNull()
                & (F.abs(F.col("left_value") - F.col("right_value")) <= F.lit(tolerance))
            )
        )
    )
    return joined.filter(~same)

Contrato estructural Bronze / Silver / Gold
Se verifica la presencia de tablas y columnas necesarias; faltantes estructurales detienen el notebook inmediatamente porque el resto de los cálculos no sería confiable.

In [0]:
missing_tables = [name for _, name in TABLE_MAP if not spark.catalog.tableExists(name)]
run_check(
    "estructura_25_tablas",
    lambda: (len(missing_tables) == 0, f"Faltantes: {missing_tables}")
)
if missing_tables:
    raise AssertionError(f"Faltan tablas requeridas: {missing_tables}. Revisá 01/02/03 y la séptima Gold.")

required_silver = {
    "orders": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
               "order_delivered_customer_date", "order_estimated_delivery_date",
               "fecha_inconsistente", "is_delivered_late"],
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"],
    "order_payments": ["order_id", "payment_sequential", "payment_type", "payment_value"],
    "customers": ["customer_id", "customer_state", "zip_geo_match"],
    "sellers": ["seller_id", "zip_geo_match"],
    "products": ["product_id", "product_category_name", "category_for_analysis"],
    "geolocation": ["geolocation_zip_code_prefix", "zip_state_ambiguous"],
}
required_gold = {
    "gold_monthly_sales": ["mes", "gmv_brl", "flete_brl", "pagos_brl", "pedidos_entregados",
                           "pedidos_con_items", "unidades_vendidas", "ticket_promedio_brl"],
    "gold_monthly_category_share": ["mes", "categoria_analitica", "gmv_brl", "unidades_vendidas", "participacion_gmv_pct"],
    "gold_monthly_top_products": ["mes", "product_id", "ranking_mes", "unidades_vendidas", "gmv_brl"],
    "gold_monthly_category_state": ["mes", "estado", "categoria_pt", "gmv_brl"],
    "gold_delivery_performance": ["mes", "estado", "nivel_geografico", "pedidos_entregados",
                                  "entregas_evaluables", "entregas_tardias",
                                  "entregas_con_fechas_faltantes", "entregas_fechas_inconsistentes",
                                  "porcentaje_tardias", "porcentaje_a_tiempo"],
    "gold_monthly_payment_methods": ["mes", "metodo_pago", "valor_pagado_brl", "porcentaje_valor_pagado"],
    "gold_monthly_portfolio_evolution": ["mes", "gmv_brl", "unidades_vendidas", "categorias_activas",
                                         "categorias_nuevas", "categorias_reactivadas",
                                         "participacion_top5_pct", "participacion_top10_pct",
                                         "hhi_portfolio", "categoria_lider", "participacion_lider_pct"]
}

missing_columns = {}
for name in BRONZE_SILVER:
    short = f"silver_{name}"
    actual = set(spark.table(f"{PREFIX}.{short}").columns)
    needed = set(KEYS[name] + ["dq_status", "dq_issues", "dq_issue_count", "silver_row_hash"] + required_silver.get(name, []))
    if needed - actual:
        missing_columns[short] = sorted(needed - actual)
for name in GOLD:
    actual = set(spark.table(f"{PREFIX}.{name}").columns)
    needed = set(GOLD_KEYS[name] + ["gold_row_hash"] + required_gold.get(name, []))
    if needed - actual:
        missing_columns[name] = sorted(needed - actual)
run_check("columnas_obligatorias", lambda: (not missing_columns, missing_columns))
if missing_columns:
    raise AssertionError(f"Columnas faltantes: {missing_columns}. Completá enriquecimientos o Gold.")

PASS | CRITICAL | estructura_25_tablas | Faltantes: []
PASS | CRITICAL | columnas_obligatorias | {}


Bronze: disponibilidad y volumen
Se exigen tablas no vacías y presencia de campos de claves de origen. No se exige igualdad Bronze/Silver: Silver resuelve duplicados y geolocalización consolida un ZIP por fila.

In [0]:
bronze_counts = {}
for name in BRONZE_SILVER:
    tbl = f"{PREFIX}.bronze_{name}"
    df = spark.table(tbl)
    n = df.count()
    bronze_counts[name] = n
    check(f"bronze_no_vacia_{name}", n > 0, f"{n:,} filas")
    missing = sorted(set(KEYS[name]) - set(df.columns))
    check(f"bronze_claves_presentes_{name}", not missing, missing)

print("Total Bronze:", sum(bronze_counts.values()))

PASS | CRITICAL | bronze_no_vacia_customers | 298,323 filas
PASS | CRITICAL | bronze_claves_presentes_customers | []
PASS | CRITICAL | bronze_no_vacia_sellers | 9,285 filas
PASS | CRITICAL | bronze_claves_presentes_sellers | []
PASS | CRITICAL | bronze_no_vacia_category_translation | 213 filas
PASS | CRITICAL | bronze_claves_presentes_category_translation | []
PASS | CRITICAL | bronze_no_vacia_orders | 298,323 filas
PASS | CRITICAL | bronze_claves_presentes_orders | []
PASS | CRITICAL | bronze_no_vacia_order_items | 337,950 filas
PASS | CRITICAL | bronze_claves_presentes_order_items | []
PASS | CRITICAL | bronze_no_vacia_order_payments | 311,658 filas
PASS | CRITICAL | bronze_claves_presentes_order_payments | []
PASS | CRITICAL | bronze_no_vacia_order_reviews | 297,672 filas
PASS | CRITICAL | bronze_claves_presentes_order_reviews | []
PASS | CRITICAL | bronze_no_vacia_products | 98,853 filas
PASS | CRITICAL | bronze_claves_presentes_products | []
PASS | CRITICAL | bronze_no_vacia_geolo

Silver: unicidad, nulos, DQ y coherencia de volumen
Los registros WARN son observaciones esperadas y no son fallas. Los rechazos se contabilizan en silver_quality_rejects. silver_geolocation es por ZIP único, no por observación geográfica original.

In [0]:
silver_counts = {}
silver_warnings = {}
for name in BRONZE_SILVER:
    df = spark.table(f"{PREFIX}.silver_{name}")
    keys = KEYS[name]
    n = df.count()
    silver_counts[name] = n
    check(f"silver_no_vacia_{name}", n > 0, f"{n:,} filas")
    run_check(f"silver_claves_validas_{name}", lambda d=df, k=keys: (key_issues(d, k) == 0, f"Claves {k}"))
    run_check(f"silver_sin_duplicados_{name}", lambda d=df, k=keys: (duplicate_keys(d, k) == 0, f"Grano {k}"))
    check(f"silver_no_supera_bronze_{name}", n <= bronze_counts[name], f"Silver={n}, Bronze={bronze_counts[name]}")

    quality = df.agg(
        F.sum(F.when(F.col("dq_status") == "WARN", 1).otherwise(0)).alias("warn"),
        F.sum(F.when(F.col("dq_issue_count") > 0, 1).otherwise(0)).alias("flag"),
        F.sum(F.when(
            F.col("dq_issue_count").isNull()
            | F.col("dq_status").isNull()
            | ((F.col("dq_issue_count") == 0) & (F.col("dq_status") != "OK"))
            | ((F.col("dq_issue_count") > 0) & (F.col("dq_status") != "WARN")), 1
        ).otherwise(0)).alias("invalid"),
        F.sum(F.when(F.col("silver_row_hash").isNull() | (F.col("silver_row_hash") == ""), 1).otherwise(0)).alias("missing_hash")
    ).first()
    silver_warnings[name] = int(quality.warn or 0)
    check(f"silver_dq_coherente_{name}", (quality.warn or 0) == (quality.flag or 0) and (quality.invalid or 0) == 0,
          f"WARN={quality.warn}, issue_count>0={quality.flag}, inválidas={quality.invalid}")
    check(f"silver_hash_informado_{name}", (quality.missing_hash or 0) == 0)

REJECT_COUNT = spark.table(REJECTS).count() if spark.catalog.tableExists(REJECTS) else 0
check("cuarentena_disponible", spark.catalog.tableExists(REJECTS),
      f"silver_quality_rejects={REJECT_COUNT:,} registros")
check("geo_resumida_por_zip", silver_counts["geolocation"] <= bronze_counts["geolocation"],
      f"Bronze observaciones={bronze_counts['geolocation']:,}; Silver ZIP={silver_counts['geolocation']:,}")

PASS | CRITICAL | silver_no_vacia_customers | 99,441 filas
PASS | CRITICAL | silver_claves_validas_customers | Claves ['customer_id']
PASS | CRITICAL | silver_sin_duplicados_customers | Grano ['customer_id']
PASS | CRITICAL | silver_no_supera_bronze_customers | Silver=99441, Bronze=298323
PASS | CRITICAL | silver_dq_coherente_customers | WARN=0, issue_count>0=0, inválidas=0
PASS | CRITICAL | silver_hash_informado_customers | 
PASS | CRITICAL | silver_no_vacia_sellers | 3,095 filas
PASS | CRITICAL | silver_claves_validas_sellers | Claves ['seller_id']
PASS | CRITICAL | silver_sin_duplicados_sellers | Grano ['seller_id']
PASS | CRITICAL | silver_no_supera_bronze_sellers | Silver=3095, Bronze=9285
PASS | CRITICAL | silver_dq_coherente_sellers | WARN=0, issue_count>0=0, inválidas=0
PASS | CRITICAL | silver_hash_informado_sellers | 
PASS | CRITICAL | silver_no_vacia_category_translation | 71 filas
PASS | CRITICAL | silver_claves_validas_category_translation | Claves ['product_category_name'

True

Silver: integridad referencial y enriquecimientos
Los FK se validan con anti-join. Para evitar una falsa exclusión, el estado comercial se toma de silver_customers, no de una coincidencia de ZIP geográfico.

In [0]:
RELATIONS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id")
]

for child, fk, parent, pk in RELATIONS:
    def check_fk(child=child, fk=fk, parent=parent, pk=pk):
        a = spark.table(f"{PREFIX}.silver_{child}").select(F.col(fk).alias("id")).filter("id IS NOT NULL").distinct()
        b = spark.table(f"{PREFIX}.silver_{parent}").select(F.col(pk).alias("id")).distinct()
        orphans = a.join(b, "id", "left_anti").count()
        return orphans == 0, f"FK distintos huérfanos={orphans}"
    run_check(f"fk_{child}_{fk}_a_{parent}", check_fk)

products = spark.table(f"{PREFIX}.silver_products")
run_check("producto_categorizacion_sin_nulos", lambda: (
    products.filter(F.col("category_for_analysis").isNull()).count() == 0,
    "Todos los productos deben tener category_for_analysis o 'Sin categoría'"
))

orders = spark.table(f"{PREFIX}.silver_orders")
run_check("pedidos_flag_tardanza", lambda: (
    orders.filter(
        F.col("order_delivered_customer_date").isNotNull()
        & F.col("order_estimated_delivery_date").isNotNull()
        & (~F.col("is_delivered_late").eqNullSafe(
            F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")
        ))
    ).count() == 0,
    "is_delivered_late debe concordar con entrega real > estimada"
))

PASS | CRITICAL | fk_orders_customer_id_a_customers | FK distintos huérfanos=0
PASS | CRITICAL | fk_order_items_order_id_a_orders | FK distintos huérfanos=0
PASS | CRITICAL | fk_order_items_product_id_a_products | FK distintos huérfanos=0
PASS | CRITICAL | fk_order_items_seller_id_a_sellers | FK distintos huérfanos=0
PASS | CRITICAL | fk_order_payments_order_id_a_orders | FK distintos huérfanos=0
PASS | CRITICAL | fk_order_reviews_order_id_a_orders | FK distintos huérfanos=0
PASS | CRITICAL | producto_categorizacion_sin_nulos | Todos los productos deben tener category_for_analysis o 'Sin categoría'
PASS | CRITICAL | pedidos_flag_tardanza | is_delivered_late debe concordar con entrega real > estimada


True

Gold: grano, hashes y reconciliación contra Silver
Se reconstruyen de manera independiente el número de pedidos entregados, las unidades y el GMV desde Silver, sin unir pagos con los ítems (esto evitaría duplicación de importes).

In [0]:
gold_counts = {}
for name in GOLD:
    df = spark.table(f"{PREFIX}.{name}")
    gold_counts[name] = df.count()
    check(f"gold_no_vacia_{name}", gold_counts[name] > 0, f"{gold_counts[name]:,} filas")
    run_check(f"gold_claves_validas_{name}", lambda d=df, k=GOLD_KEYS[name]: (key_issues(d, k) == 0, k))
    run_check(f"gold_sin_duplicados_{name}", lambda d=df, k=GOLD_KEYS[name]: (duplicate_keys(d, k) == 0, k))
    run_check(f"gold_hash_informado_{name}", lambda d=df: (
        d.filter(F.col("gold_row_hash").isNull() | (F.col("gold_row_hash") == "")).count() == 0
    ))

sales = spark.table(f"{PREFIX}.gold_monthly_sales")
category = spark.table(f"{PREFIX}.gold_monthly_category_share")
cat_state = spark.table(f"{PREFIX}.gold_monthly_category_state")
top = spark.table(f"{PREFIX}.gold_monthly_top_products")
delivery = spark.table(f"{PREFIX}.gold_delivery_performance")
payment = spark.table(f"{PREFIX}.gold_monthly_payment_methods")
portfolio = spark.table(f"{PREFIX}.gold_monthly_portfolio_evolution")

eligible_orders = (
    orders
    .filter(F.lower(F.trim(F.col("order_status"))) == "delivered")
    .withColumn("mes", F.to_date(F.date_trunc("month", F.col("order_purchase_timestamp"))))
    .filter(F.col("mes").isNotNull())
    .select("order_id", "mes")
)
expected_orders = eligible_orders.groupBy("mes").agg(F.count("*").alias("expected_orders"))
item_totals = (
    spark.table(f"{PREFIX}.silver_order_items")
    .join(eligible_orders, "order_id", "inner")
    .groupBy("mes")
    .agg(
        F.count("*").alias("expected_units"),
        F.sum("price").alias("expected_gmv"),
        F.sum("freight_value").alias("expected_freight")
    )
)
# Mantener meses con pedidos entregados pero sin líneas de artículos.
expected_items = (
    expected_orders.select("mes")
    .join(item_totals, "mes", "left")
    .fillna({"expected_units": 0, "expected_gmv": 0, "expected_freight": 0})
)

RECONCILIATIONS = [
    ("pedidos_entregados_silver_gold", sales, expected_orders, "pedidos_entregados", "expected_orders"),
    ("unidades_silver_gold", sales, expected_items, "unidades_vendidas", "expected_units"),
    ("gmv_silver_gold", sales, expected_items, "gmv_brl", "expected_gmv"),
    ("flete_silver_gold", sales, expected_items, "flete_brl", "expected_freight"),
    ("gmv_ventas_categorias", sales, category.groupBy("mes").agg(F.sum("gmv_brl").alias("total")), "gmv_brl", "total"),
    ("gmv_ventas_estados", sales, cat_state.groupBy("mes").agg(F.sum("gmv_brl").alias("total")), "gmv_brl", "total"),
    ("gmv_ventas_portfolio", sales, portfolio, "gmv_brl", "gmv_brl"),
    ("unidades_ventas_portfolio", sales, portfolio, "unidades_vendidas", "unidades_vendidas"),
    # En meses sin pago, Gold Sales contiene NULL y Gold Payment Methods puede no tener fila.
    # Completar esos meses desde Sales evita marcar como fallo la ausencia esperada de pagos.
    ("pagos_ventas_metodos", sales,
     sales.select("mes").join(
         payment.groupBy("mes").agg(F.sum("valor_pagado_brl").alias("total")),
         "mes", "left"
     ), "pagos_brl", "total"),
    ("entregas_nacional_ventas", sales, delivery.filter(F.col("nivel_geografico") == "BRASIL"),
     "pedidos_entregados", "pedidos_entregados")
]
run_check("pagos_sin_meses_extranos", lambda: (
    payment.select("mes").distinct()
    .join(sales.select("mes").distinct(), "mes", "left_anti").count() == 0
))

for label, a, b, ca, cb in RECONCILIATIONS:
    run_check(f"recon_{label}", lambda x=a, y=b, cx=ca, cy=cb: (
        compare_months(x, y, cx, cy).count() == 0, "Descuadre por mes entre ambas fuentes"
    ))

PASS | CRITICAL | gold_no_vacia_gold_monthly_sales | 23 filas
PASS | CRITICAL | gold_claves_validas_gold_monthly_sales | ['mes']
PASS | CRITICAL | gold_sin_duplicados_gold_monthly_sales | ['mes']
PASS | CRITICAL | gold_hash_informado_gold_monthly_sales | 
PASS | CRITICAL | gold_no_vacia_gold_monthly_category_share | 1,273 filas
PASS | CRITICAL | gold_claves_validas_gold_monthly_category_share | ['mes', 'categoria_analitica']
PASS | CRITICAL | gold_sin_duplicados_gold_monthly_category_share | ['mes', 'categoria_analitica']
PASS | CRITICAL | gold_hash_informado_gold_monthly_category_share | 
PASS | CRITICAL | gold_no_vacia_gold_monthly_top_products | 212 filas
PASS | CRITICAL | gold_claves_validas_gold_monthly_top_products | ['mes', 'ranking_mes']
PASS | CRITICAL | gold_sin_duplicados_gold_monthly_top_products | ['mes', 'ranking_mes']
PASS | CRITICAL | gold_hash_informado_gold_monthly_top_products | 
PASS | CRITICAL | gold_no_vacia_gold_monthly_category_state | 11,672 filas
PASS | CRITIC

Gold: porcentajes, rankings, operaciones y concentración
Los porcentajes de órdenes que usan cada método de pago no tienen que sumar 100%: una orden puede combinar métodos. La métrica de puntualidad utiliza solo las entregas evaluables.

In [0]:
# Validación de la participación por categoría en cada mes.
run_check("categoria_participacion_recalcula", lambda: (
    category.join(sales.select("mes", F.col("gmv_brl").alias("gmv_ventas")), "mes")
    .filter(
        (F.col("gmv_ventas") > 0)
        & (F.abs(
            F.col("participacion_gmv_pct")
            - F.round(F.col("gmv_brl") / F.col("gmv_ventas") * 100, 4)
        ) > F.lit(0.02))
    ).count() == 0,
    "Participación basada en GMV de artículos, incluyendo Sin categoría"
))

run_check("top10_rank_y_unicidad", lambda: (
    top.filter((F.col("ranking_mes") < 1) | (F.col("ranking_mes") > 10)).count() == 0
    and top.groupBy("mes").agg(F.count("*").alias("n"),
                                F.countDistinct("ranking_mes").alias("rangos"))
           .filter((F.col("n") > 10) | (F.col("n") != F.col("rangos"))).count() == 0,
    "Cada mes tiene entre 1 y 10 productos rankeados, sin ranks duplicados"
))

run_check("delivery_particion_evaluable", lambda: (
    delivery.filter(
        (F.col("entregas_tardias") > F.col("entregas_evaluables"))
        | (F.col("entregas_evaluables") > F.col("pedidos_entregados"))
        | (F.col("entregas_evaluables") + F.col("entregas_con_fechas_faltantes")
           + F.col("entregas_fechas_inconsistentes") != F.col("pedidos_entregados"))
    ).count() == 0,
    "Evaluables + sin fechas + fechas inconsistentes = entregados"
))

run_check("delivery_tasa_tardias", lambda: (
    delivery.filter(
        (F.col("entregas_evaluables") > 0)
        & (
            (F.abs(F.col("porcentaje_tardias") - F.round(
                100 * F.col("entregas_tardias") / F.col("entregas_evaluables"), 2)) > 0.01)
            | (F.abs(F.col("porcentaje_a_tiempo") - F.round(
                100 * (F.col("entregas_evaluables") - F.col("entregas_tardias"))
                / F.col("entregas_evaluables"), 2)) > 0.01)
        )
    ).count() == 0
))

# Reconciliación estados → BRASIL por MES y CONTEOS (no duplicar nivel geográfico).
for metric in ["pedidos_entregados", "entregas_evaluables", "entregas_tardias",
               "entregas_con_fechas_faltantes", "entregas_fechas_inconsistentes"]:
    state_metric = delivery.filter(F.col("nivel_geografico") == "ESTADO")\
        .groupBy("mes").agg(F.sum(metric).alias("total_estado"))
    national_metric = delivery.filter(F.col("nivel_geografico") == "BRASIL")
    run_check(f"delivery_estado_brasil_{metric}", lambda st=state_metric, br=national_metric, m=metric: (
        compare_months(br, st, m, "total_estado").count() == 0
    ))

run_check("payments_participacion_valor", lambda: (
    payment.filter(F.col("valor_pagado_total_mes_brl") > 0)
    .groupBy("mes")
    .agg(F.sum("porcentaje_valor_pagado").alias("total_pct"))
    .filter(F.abs(F.col("total_pct") - 100) > 0.05)
    .count() == 0,
    "Solo porcentaje de importe; porcentaje de pedidos puede sumar >100%"
))

run_check("portfolio_rangos_concentracion", lambda: (
    portfolio.filter(
        (F.col("categorias_activas") < 0)
        | (F.col("categorias_nuevas") < 0)
        | (F.col("categorias_reactivadas") < 0)
        | (F.col("categorias_nuevas") > F.col("categorias_activas"))
        | (F.col("participacion_top5_pct") < 0)
        | (F.col("participacion_top5_pct") > 100)
        | (F.col("participacion_top10_pct") < 0)
        | (F.col("participacion_top10_pct") > 100)
        | (F.col("participacion_top5_pct") > F.col("participacion_top10_pct"))
        | (F.col("hhi_portfolio") < 0)
        | (F.col("hhi_portfolio") > 10000)
        | (F.col("participacion_lider_pct") < 0)
        | (F.col("participacion_lider_pct") > 100)
    ).count() == 0,
    "HHI de 0 a 10000; Top 5 <= Top 10"
))

# Cálculo independiente de Top 5 / Top 10 / HHI para detectar errores de fórmula.
w_mes_categoria = Window.partitionBy("mes")
w_rank_categoria = Window.partitionBy("mes").orderBy(
    F.desc("gmv_brl"), F.asc("categoria_analitica")
)
recalculated = (
    category
    .withColumn("total_mes", F.sum("gmv_brl").over(w_mes_categoria))
    .withColumn("pos", F.row_number().over(w_rank_categoria))
    .groupBy("mes")
    .agg(
        F.round(100 * F.sum(F.when(F.col("pos") <= 5, F.col("gmv_brl")).otherwise(0))
                / F.max("total_mes"), 2).alias("top5_recalc"),
        F.round(100 * F.sum(F.when(F.col("pos") <= 10, F.col("gmv_brl")).otherwise(0))
                / F.max("total_mes"), 2).alias("top10_recalc"),
        F.round(10000 * F.sum(F.pow(F.col("gmv_brl") / F.col("total_mes"), 2)), 2)
            .alias("hhi_recalc")
    )
)
run_check("portfolio_top5_top10_hhi_recalcula", lambda: (
    portfolio.join(recalculated, "mes", "full_outer")
    .filter(
        F.col("top5_recalc").isNull()
        | F.col("participacion_top5_pct").isNull()
        | (F.abs(F.col("participacion_top5_pct") - F.col("top5_recalc")) > 0.02)
        | (F.abs(F.col("participacion_top10_pct") - F.col("top10_recalc")) > 0.02)
        | (F.abs(F.col("hhi_portfolio") - F.col("hhi_recalc")) > 0.02)
    ).count() == 0,
    "HHI y concentración independientes, tolerancia 0.02"
))

PASS | CRITICAL | categoria_participacion_recalcula | Participación basada en GMV de artículos, incluyendo Sin categoría
PASS | CRITICAL | top10_rank_y_unicidad | Cada mes tiene entre 1 y 10 productos rankeados, sin ranks duplicados
PASS | CRITICAL | delivery_particion_evaluable | Evaluables + sin fechas + fechas inconsistentes = entregados
PASS | CRITICAL | delivery_tasa_tardias | 
PASS | CRITICAL | delivery_estado_brasil_pedidos_entregados | 
PASS | CRITICAL | delivery_estado_brasil_entregas_evaluables | 
PASS | CRITICAL | delivery_estado_brasil_entregas_tardias | 
PASS | CRITICAL | delivery_estado_brasil_entregas_con_fechas_faltantes | 
PASS | CRITICAL | delivery_estado_brasil_entregas_fechas_inconsistentes | 
PASS | CRITICAL | payments_participacion_valor | Solo porcentaje de importe; porcentaje de pedidos puede sumar >100%
PASS | CRITICAL | portfolio_rangos_concentracion | HHI de 0 a 10000; Top 5 <= Top 10
PASS | CRITICAL | portfolio_top5_top10_hhi_recalcula | HHI y concentración 

True

## Alertas de duplicados — lote Bronze reciente

Se consulta la auditoría de Silver para el lote más reciente. Los duplicados idénticos y versiones conflictivas
se notifican como **WARNING** sin bloquear el Job: el fallo crítico sería que llegaran duplicados a las claves Silver
(o que Gold dejara de reconciliar). Geolocalización se mide por payload repetido, no por ZIP observado.

In [0]:
if spark.catalog.tableExists(DUPLICATE_ALERTS) and spark.catalog.tableExists(BRONZE_AUDIT):
    latest_batches = (spark.table(BRONZE_AUDIT)
        .groupBy("batch_id").agg(F.max("ingestion_timestamp").alias("last_run"))
        .orderBy(F.desc("last_run")).limit(1).collect())
    latest_batch_id = latest_batches[0]["batch_id"] if latest_batches else None
    if latest_batch_id:
        events = spark.table(DUPLICATE_ALERTS).filter(F.col("bronze_batch_id") == latest_batch_id)
        # Evitar sumar ejecuciones Silver anteriores sobre el mismo batch.
        latest_run = (events.groupBy("silver_run_id")
            .agg(F.max("recorded_at").alias("last_silver"))
            .orderBy(F.desc("last_silver")).limit(1).collect())
        if latest_run:
            events = events.filter(F.col("silver_run_id") == latest_run[0]["silver_run_id"])
            # Geolocation contiene observaciones repetidas legítimas por ZIP/payload;
            # no mezclarlas con duplicados de claves de pedidos/productos/clientes.
            business_events = events.filter(F.col("source_table") != "geolocation")
            total = business_events.agg(
                F.sum("rows_received").alias("received"),
                F.sum("repeated_key_rows").alias("repeated"),
                F.sum("exact_duplicate_rows").alias("exact"),
                F.sum("conflicting_version_rows").alias("conflict")
            ).first()
            incoming = int(total["received"] or 0)
            duplicate_rows = int(total["repeated"] or 0)
            conflict_rows = int(total["conflict"] or 0)
            display(events.orderBy("source_table"))
            geo_repeats = (events.filter(F.col("source_table") == "geolocation")
                           .agg(F.sum("repeated_key_rows").alias("n")).first()["n"] or 0)
            check("repeticiones_geo_informativas", True,
                  f"Observaciones geográficas repetidas: {int(geo_repeats)}", severity="INFO")
            check("alerta_duplicados_lote", duplicate_rows == 0,
                  f"Lote={latest_batch_id}; recibidos={incoming}; duplicados={duplicate_rows}; "
                  f"idénticos={int(total['exact'] or 0)}; conflictos={conflict_rows}",
                  severity="WARNING")
        else:
            check("alerta_duplicados_no_registrada", False,
                  "La auditoría Silver no contiene este batch", severity="WARNING")
else:
    check("auditoria_duplicados_disponible", False,
          "Falta silver_duplicate_alerts; ejecutar primero 02_build_silver_v2", severity="WARNING")


silver_run_id,bronze_batch_id,source_table,rows_received,repeated_key_rows,exact_duplicate_rows,conflicting_version_rows,new_key_rows,alert_level,recorded_at
f00694c8-c46a-4c60-91a0-1014a4a21ec4,1f7d850d-bb2c-49c3-957a-907c005350db,category_translation,71,71,71,0,0,WARN_DUPLICATES,2026-10-25T19:29:57.512Z
f00694c8-c46a-4c60-91a0-1014a4a21ec4,1f7d850d-bb2c-49c3-957a-907c005350db,customers,99441,99441,99441,0,0,WARN_DUPLICATES,2026-10-25T19:29:12.064Z
f00694c8-c46a-4c60-91a0-1014a4a21ec4,1f7d850d-bb2c-49c3-957a-907c005350db,geolocation,1000163,1000163,1000163,0,0,INFO_GEO_REPEATED,2026-10-25T19:31:57.185Z
f00694c8-c46a-4c60-91a0-1014a4a21ec4,1f7d850d-bb2c-49c3-957a-907c005350db,order_items,112650,112650,112650,0,0,WARN_DUPLICATES,2026-10-25T19:30:37.792Z
f00694c8-c46a-4c60-91a0-1014a4a21ec4,1f7d850d-bb2c-49c3-957a-907c005350db,order_payments,103886,103886,103886,0,0,WARN_DUPLICATES,2026-10-25T19:30:57.282Z
f00694c8-c46a-4c60-91a0-1014a4a21ec4,1f7d850d-bb2c-49c3-957a-907c005350db,order_reviews,99224,99224,99224,0,0,WARN_DUPLICATES,2026-10-25T19:31:16.915Z
f00694c8-c46a-4c60-91a0-1014a4a21ec4,1f7d850d-bb2c-49c3-957a-907c005350db,orders,99441,99441,99441,0,0,WARN_DUPLICATES,2026-10-25T19:30:16.340Z
f00694c8-c46a-4c60-91a0-1014a4a21ec4,1f7d850d-bb2c-49c3-957a-907c005350db,products,32951,32951,32951,0,0,WARN_DUPLICATES,2026-10-25T19:31:37.559Z
f00694c8-c46a-4c60-91a0-1014a4a21ec4,1f7d850d-bb2c-49c3-957a-907c005350db,sellers,3095,3095,3095,0,0,WARN_DUPLICATES,2026-10-25T19:29:38.748Z


PASS | INFO     | repeticiones_geo_informativas | Observaciones geográficas repetidas: 1000163
FAIL | WARNING  | alerta_duplicados_lote | Lote=1f7d850d-bb2c-49c3-957a-907c005350db; recibidos=550759; duplicados=550759; idénticos=550759; conflictos=0


Snapshots determinísticos (conteos + fingerprint)
Se calcula una firma independiente del orden de filas de cada una de las 25 tablas. Usa SHA-256 del contenido JSON con nombres de columna ordenados; se combinan conteo, suma exacta de un fragmento del hash, mínimo y máximo. Es una huella probabilística de auditoría, no una demostración matemática de igualdad de todos los bytes.

Rendimiento: esta es la celda más costosa porque lee todas las tablas, incluida la geolocalización Bronze (≈1 millón de filas). Permite comparar cambios de contenido y no solo conteos.


In [0]:
def content_fingerprint(df):
    cols = sorted(df.columns)
    payload = F.to_json(
        F.struct(*[F.col(f"`{c.replace('`', '``')}`").alias(c) for c in cols]),
        options={"ignoreNullFields": "false"}
    )
    row_hash = F.sha2(payload, 256)
    # Primeros 15 dígitos hex = 60 bits: sumarlos como DECIMAL(38,0) evita overflow
    # incluso al agregar muchos millones de registros.
    fragment = F.conv(F.substring(row_hash, 1, 15), 16, 10).cast("decimal(38,0)")
    metrics = df.agg(
        F.count("*").alias("n"),
        F.sum(fragment).alias("sum_part"),
        F.min(row_hash).alias("min_hash"),
        F.max(row_hash).alias("max_hash")
    ).first()
    signature = "|".join([
        str(metrics.n), str(metrics.sum_part or 0),
        str(metrics.min_hash or ""), str(metrics.max_hash or "")
    ])
    return int(metrics.n), hashlib.sha256(signature.encode("utf-8")).hexdigest()

snapshots_current = []
for layer, table_name in TABLE_MAP:
    print(f"Fingerprint: {layer:<6} {table_name}")
    n, digest = content_fingerprint(spark.table(table_name))
    snapshots_current.append((RUN_ID, layer, table_name, n, digest))
    check(f"snapshot_coherente_{table_name.split('.')[-1]}",
          n == (bronze_counts.get(table_name.split("bronze_")[-1]) if layer == "BRONZE" else
                silver_counts.get(table_name.split("silver_")[-1]) if layer == "SILVER" else
                gold_counts.get(table_name.split(".")[-1])),
          f"filas={n}, fingerprint={digest[:16]}…")

Fingerprint: BRONZE workspace.olist_lakehouse.bronze_customers
PASS | CRITICAL | snapshot_coherente_bronze_customers | filas=298323, fingerprint=7be26cf796a1455e…
Fingerprint: BRONZE workspace.olist_lakehouse.bronze_sellers
PASS | CRITICAL | snapshot_coherente_bronze_sellers | filas=9285, fingerprint=8f75a10739f8fe85…
Fingerprint: BRONZE workspace.olist_lakehouse.bronze_category_translation
PASS | CRITICAL | snapshot_coherente_bronze_category_translation | filas=213, fingerprint=1972e7bd33d5d9ca…
Fingerprint: BRONZE workspace.olist_lakehouse.bronze_orders
PASS | CRITICAL | snapshot_coherente_bronze_orders | filas=298323, fingerprint=f6d3332550a2c9d2…
Fingerprint: BRONZE workspace.olist_lakehouse.bronze_order_items
PASS | CRITICAL | snapshot_coherente_bronze_order_items | filas=337950, fingerprint=07d9e4435a2cf0f6…
Fingerprint: BRONZE workspace.olist_lakehouse.bronze_order_payments
PASS | CRITICAL | snapshot_coherente_bronze_order_payments | filas=311658, fingerprint=357d19a732d36995…
F

Auditoría persistente y comparación automática entre ejecuciones

En `auto` se busca la **última corrida aprobada con 25 snapshots completos**, se comparan firmas de contenido y se clasifica la situación. No se borra ni modifica la auditoría previa. Los demás modos siguen disponibles para pruebas dirigidas.


In [0]:
AUDIT_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("job_run_id", StringType(), True),
    StructField("mode", StringType(), False),
    StructField("baseline_run_id", StringType(), True),
    StructField("recorded_at", TimestampType(), False),
    StructField("bronze_rows", LongType(), False),
    StructField("silver_rows", LongType(), False),
    StructField("gold_rows", LongType(), False),
    StructField("silver_warning_rows", LongType(), False),
    StructField("silver_rejected_rows", LongType(), False),
    StructField("gmv_total_brl", DecimalType(28, 2), True),
    StructField("checks_total", IntegerType(), False),
    StructField("checks_failed", IntegerType(), False),
    StructField("idempotence_compared", BooleanType(), False),
    StructField("idempotence_passed", BooleanType(), True),
    StructField("incremental_compared", BooleanType(), False),
    StructField("incremental_evidence", BooleanType(), True),
    StructField("status", StringType(), False)
])
CHECK_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("passed", BooleanType(), False),
    StructField("severity", StringType(), False),
    StructField("detail", StringType(), True),
    StructField("recorded_at", TimestampType(), False)
])
SNAPSHOT_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("row_count", LongType(), False),
    StructField("content_fingerprint", StringType(), False),
    StructField("recorded_at", TimestampType(), False)
])


def ensure_delta_table(name, schema):
    if not spark.catalog.tableExists(name):
        spark.createDataFrame([], schema).write.format("delta").saveAsTable(name)

ensure_delta_table(AUDIT, AUDIT_SCHEMA)
ensure_delta_table(CHECK_DETAIL, CHECK_SCHEMA)
ensure_delta_table(SNAPSHOTS, SNAPSHOT_SCHEMA)

idempotence_compared = False
idempotence_passed = None
incremental_compared = False
incremental_evidence = None
comparison_rows = []
AUTO_OUTCOME = "NO_APLICA"

# En modo automático tomamos la ÚLTIMA corrida PASS que tenga 25 snapshots
# realmente persistidos. No depende de ids ingresados manualmente.
if MODE == "auto":
    eligible_snapshots = (
        spark.table(SNAPSHOTS)
        .groupBy("run_id")
        .agg(F.countDistinct("table_name").alias("snapshot_tables"))
        .filter(F.col("snapshot_tables") == len(TABLE_MAP))
        .select("run_id")
    )
    latest_successful = (
        spark.table(AUDIT)
        .filter(F.col("status") == "PASS")
        .join(eligible_snapshots, "run_id", "inner")
        .orderBy(F.desc("recorded_at"), F.desc("run_id"))
        .limit(1)
        .collect()
    )
    BASELINE_ID = latest_successful[0]["run_id"] if latest_successful else ""
    print(f"AUTO | Baseline seleccionado: {BASELINE_ID or '(ninguno: primera corrida)'}")

if MODE in ("verify", "incremental_observe", "auto"):
    valid_id = bool(BASELINE_ID) and BASELINE_ID != RUN_ID
    # Primera ejecución del modo auto: registra un baseline sin marcar fallo.
    if MODE == "auto" and not valid_id:
        AUTO_OUTCOME = "PRIMER_BASELINE"
        check("auto_primera_ejecucion", True,
              "Sin baseline aprobado previo con 25 snapshots; se registra el estado actual", severity="INFO")
    else:
        check("baseline_id_indicado", valid_id,
              f"baseline_run_id={BASELINE_ID or '(vacío)'}")
        prior = (spark.table(AUDIT)
                 .filter(F.col("run_id") == BASELINE_ID)
                 .orderBy(F.desc("recorded_at")).limit(1).collect()) if valid_id else []
        good_prior = bool(prior) and prior[0]["status"] == "PASS"
        check("baseline_existe_y_aprobado", good_prior,
              f"Encontrado={bool(prior)}, status={prior[0]['status'] if prior else None}")

        if good_prior:
            previous = {
                r["table_name"]: (int(r["row_count"]), r["content_fingerprint"])
                for r in spark.table(SNAPSHOTS).filter(F.col("run_id") == BASELINE_ID).collect()
            }
            actual = {name: (n, fp) for _, _, name, n, fp in snapshots_current}
            check("baseline_contiene_25_snapshots", len(previous) == len(TABLE_MAP),
                  f"Snapshots previos={len(previous)}; requeridos={len(TABLE_MAP)}")
            for layer, t in TABLE_MAP:
                old = previous.get(t)
                now = actual.get(t)
                if old is not None and now is not None:
                    comparison_rows.append((
                        layer, t, old[0], now[0], now[0] - old[0], old != now
                    ))

            if MODE == "auto":
                bronze_changes = [r for r in comparison_rows if r[0] == "BRONZE" and r[5]]
                silver_changes = [r for r in comparison_rows if r[0] == "SILVER" and r[5]]
                gold_changes = [r for r in comparison_rows if r[0] == "GOLD" and r[5]]
                bronze_grew = any(r[0] == "BRONZE" and r[4] > 0 for r in comparison_rows)
                bronze_shrank = any(r[0] == "BRONZE" and r[4] < 0 for r in comparison_rows)
                scope_names = [t for layer, t in TABLE_MAP if layer in ("SILVER", "GOLD")]
                analytic_equal = (len(previous) == len(TABLE_MAP)
                                  and all(previous.get(t) == actual.get(t) for t in scope_names))
                check("auto_bronze_sin_reducciones", not bronze_shrank,
                      f"Tablas Bronze con reducción de filas: "
                      f"{[r[1] for r in comparison_rows if r[0] == 'BRONZE' and r[4] < 0]}")

                if analytic_equal:
                    # Aun cuando crezca Bronze por reingesta, Silver y Gold deben quedar iguales.
                    idempotence_compared = True
                    idempotence_passed = True
                    AUTO_OUTCOME = "RECARGA_DUPLICADA_DEPURADA" if bronze_changes else "REPROCESO_SIN_CAMBIOS"
                    check("auto_idempotencia_silver_gold", True,
                          f"16/16 tablas Silver+Gold idénticas; "
                          f"Bronze cambió en {len(bronze_changes)} tabla(s)")
                elif not bronze_changes:
                    # Sin fuente modificada, cualquier cambio analítico es anómalo.
                    idempotence_compared = True
                    idempotence_passed = False
                    AUTO_OUTCOME = "CAMBIO_ANALITICO_INESPERADO"
                    check("auto_idempotencia_silver_gold", False,
                          f"Bronze no cambió, pero cambiaron {len(silver_changes)} Silver "
                          f"y {len(gold_changes)} Gold")
                else:
                    # Cambió el origen: no exigir igualdad Silver/Gold, pues pueden existir
                    # ventas nuevas. La coherencia se controla con las reconciliaciones previas.
                    incremental_compared = True
                    sound_incremental = (bronze_grew and not bronze_shrank and bool(silver_changes)
                                         and not (bool(gold_changes) and not bool(silver_changes)))
                    incremental_evidence = bool(sound_incremental)
                    AUTO_OUTCOME = ("CAMBIO_INCREMENTAL_CONCILIADO" if sound_incremental
                                    else "CAMBIO_INCREMENTAL_INCONSISTENTE")
                    check("auto_incremental_origen_y_destino", sound_incremental,
                          f"Bronze creció={bronze_grew}; tablas Silver modificadas="
                          f"{len(silver_changes)}; tablas Gold modificadas={len(gold_changes)}. "
                          "Los controles de negocio anteriores verifican las reconciliaciones.")
                    check("auto_idempotencia_no_aplicable_con_nuevos_datos", True,
                          "Silver o Gold cambió junto con Bronze: se evalúa incrementalidad, "
                          "no igualdad de fingerprints", severity="INFO")
                print(f"AUTO | Resultado: {AUTO_OUTCOME}")
                print(f"AUTO | Cambios por capa: Bronze={len(bronze_changes)}, "
                      f"Silver={len(silver_changes)}, Gold={len(gold_changes)}")

            elif MODE == "verify":
                idempotence_compared = True
                scope = TABLE_MAP if IDEMPOTENCE_SCOPE == "all_25" else [
                    (layer, t) for layer, t in TABLE_MAP if layer in ("SILVER", "GOLD")
                ]
                idempotence_passed = len(previous) == len(TABLE_MAP) and all(
                    previous.get(t) == actual.get(t) for _, t in scope
                )
                check("idempotencia_" + IDEMPOTENCE_SCOPE, idempotence_passed,
                      f"Conteo y fingerprint iguales en {len(scope)} tablas del alcance elegido")
                bronze_changed = any(layer == "BRONZE" and changed
                                     for layer, _, _, _, _, changed in comparison_rows)
                check("bronze_cambio_informativo", True,
                      f"Bronze cambió: {bronze_changed} (admitido si se recargó)", severity="INFO")
            else:
                incremental_compared = True
                bronze_grew = any(layer == "BRONZE" and delta > 0
                                  for layer, _, _, _, delta, _ in comparison_rows)
                silver_changed = any(layer == "SILVER" and changed
                                     for layer, _, _, _, _, changed in comparison_rows)
                gold_changed = any(layer == "GOLD" and changed
                                   for layer, _, _, _, _, changed in comparison_rows)
                no_bronze_shrink = all(layer != "BRONZE" or delta >= 0
                                       for layer, _, _, _, delta, _ in comparison_rows)
                check("incremental_bronze_filas_nuevas", bronze_grew,
                      "Debe existir al menos una fuente Bronze con más filas que el baseline")
                check("incremental_bronze_sin_borrados", no_bronze_shrink,
                      "La prueba es de adición de archivos, no corrección con borrado")
                if EXPECT_NEW_BUSINESS_KEYS:
                    check("incremental_cambio_silver", silver_changed,
                          "Se esperaban nuevas claves que debían propagarse a Silver")
                else:
                    check("incremental_silver_puede_permanecer_igual", True,
                          f"Silver cambió: {silver_changed}; duplicados pueden dejarlo igual", severity="INFO")
                if EXPECT_GOLD_CHANGE:
                    check("incremental_cambio_gold", gold_changed,
                          "Exigido: nuevos pedidos entregados con impacto comercial")
                else:
                    check("incremental_cambio_gold_no_exigido", True,
                          f"Cambio Gold observado: {gold_changed}", severity="INFO")
                incremental_evidence = (bronze_grew and no_bronze_shrink
                                        and (silver_changed or not EXPECT_NEW_BUSINESS_KEYS)
                                        and (gold_changed or not EXPECT_GOLD_CHANGE))

if comparison_rows:
    comparison_schema = ["capa", "tabla", "filas_baseline", "filas_actuales", "delta_filas", "contenido_cambio"]
    display(spark.createDataFrame(comparison_rows, comparison_schema)
            .orderBy("capa", "tabla"))

AUTO | Baseline seleccionado: 577358cb-e039-45a0-bf9b-04fdbbff614d
PASS | CRITICAL | baseline_id_indicado | baseline_run_id=577358cb-e039-45a0-bf9b-04fdbbff614d
PASS | CRITICAL | baseline_existe_y_aprobado | Encontrado=True, status=PASS
PASS | CRITICAL | baseline_contiene_25_snapshots | Snapshots previos=25; requeridos=25
PASS | CRITICAL | auto_bronze_sin_reducciones | Tablas Bronze con reducción de filas: []
PASS | CRITICAL | auto_idempotencia_silver_gold | 16/16 tablas Silver+Gold idénticas; Bronze cambió en 0 tabla(s)
AUTO | Resultado: REPROCESO_SIN_CAMBIOS
AUTO | Cambios por capa: Bronze=0, Silver=0, Gold=0


capa,tabla,filas_baseline,filas_actuales,delta_filas,contenido_cambio
BRONZE,workspace.olist_lakehouse.bronze_category_translation,213,213,0,false
BRONZE,workspace.olist_lakehouse.bronze_customers,298323,298323,0,false
BRONZE,workspace.olist_lakehouse.bronze_geolocation,3000489,3000489,0,false
BRONZE,workspace.olist_lakehouse.bronze_order_items,337950,337950,0,false
BRONZE,workspace.olist_lakehouse.bronze_order_payments,311658,311658,0,false
BRONZE,workspace.olist_lakehouse.bronze_order_reviews,297672,297672,0,false
BRONZE,workspace.olist_lakehouse.bronze_orders,298323,298323,0,false
BRONZE,workspace.olist_lakehouse.bronze_products,98853,98853,0,false
BRONZE,workspace.olist_lakehouse.bronze_sellers,9285,9285,0,false
GOLD,workspace.olist_lakehouse.gold_delivery_performance,579,579,0,false


Persistir evidencias, mostrar resultado y fallar si corresponde
Los fallos críticos provocan AssertionError después de escribir los resultados. Así el Job queda rojo y la auditoría registra qué pasó. Un INFO no bloquea el Job.

In [0]:
now = datetime.now(timezone.utc)
failed_critical = [name for name, passed, severity, _ in checks if not passed and severity == "CRITICAL"]
final_status = "FAIL" if failed_critical else "PASS"

# Snapshot de las 25 tablas observadas en esta corrida.
snapshot_rows = [(rid, layer, table, int(n), fp, now)
                 for rid, layer, table, n, fp in snapshots_current]
spark.createDataFrame(snapshot_rows, SNAPSHOT_SCHEMA).write.format("delta").mode("append").saveAsTable(SNAPSHOTS)

# Detalle auditable por control.
check_rows = [(RUN_ID, name, bool(passed), severity, detail, now)
              for name, passed, severity, detail in checks]
spark.createDataFrame(check_rows, CHECK_SCHEMA).write.format("delta").mode("append").saveAsTable(CHECK_DETAIL)

gmv = sales.agg(F.sum("gmv_brl").cast("decimal(28,2)").alias("total")).first()["total"]
audit_row = [(
    RUN_ID, JOB_RUN_ID, MODE, BASELINE_ID or None, now,
    int(sum(bronze_counts.values())), int(sum(silver_counts.values())),
    int(sum(gold_counts.values())), int(sum(silver_warnings.values())),
    int(REJECT_COUNT), gmv, int(len(checks)), int(len(failed_critical)),
    bool(idempotence_compared), idempotence_passed,
    bool(incremental_compared), incremental_evidence, final_status
)]
spark.createDataFrame(audit_row, AUDIT_SCHEMA).write.format("delta").mode("append").saveAsTable(AUDIT)

print("=" * 80)
print("VALIDACIÓN DEL LAKEHOUSE OLIST")
print("RUN_ID:", RUN_ID)
print("ESTADO:", final_status)
print("CONTROLES:", len(checks), "| CRÍTICOS FALLIDOS:", len(failed_critical))
print("BRONZE:", sum(bronze_counts.values()), "| SILVER:", sum(silver_counts.values()),
      "| GOLD:", sum(gold_counts.values()))
print("SILVER WARN:", sum(silver_warnings.values()), "| RECHAZOS:", REJECT_COUNT)
print("GMV (BRL):", gmv)
print("AUTO RESULTADO:", AUTO_OUTCOME if MODE == "auto" else "NO_APLICA")
print("BASELINE COMPARADO:", BASELINE_ID or "(sin baseline previo)")
print("IDEMPOTENCIA COMPARADA:", idempotence_compared,
      "| PASÓ:", idempotence_passed)
print("INCREMENTAL COMPARADA:", incremental_compared,
      "| EVIDENCIA:", incremental_evidence)
print("=" * 80)

display(
    spark.table(CHECK_DETAIL).filter(F.col("run_id") == RUN_ID)
    .select("check_name", "passed", "severity", "detail")
    .orderBy(F.asc("passed"), "severity", "check_name")
)
display(
    spark.table(AUDIT).orderBy(F.desc("recorded_at")).limit(10)
)

if failed_critical:
    raise AssertionError(f"Fallaron {len(failed_critical)} controles críticos: {failed_critical}")

print(f"VALIDACIONES COMPLETADAS. RUN_ID auditado: {RUN_ID}")

VALIDACIÓN DEL LAKEHOUSE OLIST
RUN_ID: 0e605f32-3f9c-496e-9b4c-aaaa89842467
ESTADO: PASS
CONTROLES: 167 | CRÍTICOS FALLIDOS: 0
BRONZE: 4652766 | SILVER: 569774 | GOLD: 13867
SILVER WARN: 2037 | RECHAZOS: 0
GMV (BRL): 13221498.11
AUTO RESULTADO: REPROCESO_SIN_CAMBIOS
BASELINE COMPARADO: 577358cb-e039-45a0-bf9b-04fdbbff614d
IDEMPOTENCIA COMPARADA: True | PASÓ: True
INCREMENTAL COMPARADA: False | EVIDENCIA: None


check_name,passed,severity,detail
alerta_duplicados_lote,false,WARNING,Lote=1f7d850d-bb2c-49c3-957a-907c005350db; recibidos=550759; duplicados=550759; idénticos=550759; conflictos=0
auto_bronze_sin_reducciones,true,CRITICAL,Tablas Bronze con reducción de filas: []
auto_idempotencia_silver_gold,true,CRITICAL,16/16 tablas Silver+Gold idénticas; Bronze cambió en 0 tabla(s)
baseline_contiene_25_snapshots,true,CRITICAL,Snapshots previos=25; requeridos=25
baseline_existe_y_aprobado,true,CRITICAL,"Encontrado=True, status=PASS"
baseline_id_indicado,true,CRITICAL,baseline_run_id=577358cb-e039-45a0-bf9b-04fdbbff614d
bronze_claves_presentes_category_translation,true,CRITICAL,[]
bronze_claves_presentes_customers,true,CRITICAL,[]
bronze_claves_presentes_geolocation,true,CRITICAL,[]
bronze_claves_presentes_order_items,true,CRITICAL,[]


run_id,job_run_id,mode,baseline_run_id,recorded_at,bronze_rows,silver_rows,gold_rows,silver_warning_rows,silver_rejected_rows,gmv_total_brl,checks_total,checks_failed,idempotence_compared,idempotence_passed,incremental_compared,incremental_evidence,status
0e605f32-3f9c-496e-9b4c-aaaa89842467,interactive,auto,577358cb-e039-45a0-bf9b-04fdbbff614d,2026-10-25T19:58:24.790Z,4652766,569774,13867,2037,0,13221498.11,167,0,true,true,false,null,PASS
577358cb-e039-45a0-bf9b-04fdbbff614d,interactive,baseline,null,2026-10-25T19:36:58.813Z,4652766,569774,13867,2037,0,13221498.11,162,0,false,null,false,null,PASS
05c975d4-c85e-4219-8fb7-3bb51cb73113,interactive,baseline,null,2026-10-25T19:12:14.774Z,3101844,569774,13867,2037,0,13221498.11,162,0,false,null,false,null,PASS
bb7b3e65-de31-4ee8-9086-2efd0640ebe6,interactive,baseline,null,2026-10-25T18:02:05.983Z,1550922,569774,13867,2037,0,13221498.11,160,0,false,null,false,null,PASS


VALIDACIONES COMPLETADAS. RUN_ID auditado: 0e605f32-3f9c-496e-9b4c-aaaa89842467
